# RoadScan run 21 — frozen-model country evaluation and error analysis

Evaluate the adopted **run17_best_finetuned.pt** without training. Keep confidence **0.46**, resolution **640**, NMS IoU **0.7**, and max_det **300** fixed. Local photos remain untouched.

China MotorBike is previously used development validation. United States is a country excluded from our fine-tuning, now evaluated once with frozen settings. Japan, India and Czech are represented in fine-tuning; up to 500 deterministic images per country will be selected outside the exact team training sample, if available. These are image-held-out diagnostics, not unseen-country tests. External checkpoint pretraining overlap is unknown for all countries.

We remove exact training-image duplicates by SHA-256 and exact evaluation duplicates across cohorts. This cannot rule out visually similar frames or route leakage because drive IDs/GPS are unavailable. Sample selection uses filenames only, before predictions, with no class enrichment. Different country class mixes make aggregate scores descriptive rather than a controlled causal estimate of country effects.

## 1. Freeze the model and evaluation protocol
Verify the exact checkpoint, pin dependencies, and record the operating point before opening the country results. The study does not change the adopted model or tune parameters on the test country.

In [ ]:
from pathlib import Path
import sys,subprocess,json,hashlib,math,shutil,time,gc
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image,ImageDraw
import yaml
from IPython.display import display,Markdown
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch,ultralytics
from ultralytics import YOLO
assert ultralytics.__version__=='8.4.177'
DEVICE=0 if torch.cuda.is_available() else 'cpu'
BASE=Path('/kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning')
SOURCE=Path('/kaggle/input/datasets/sarvii7/roadscan-yolo/RoadScan_yolo')
assert SOURCE.is_dir(),'Attach Sara roadscan-yolo dataset.'
WEIGHTS=BASE/'run17_model_comparison/run17_best_finetuned.pt'
SHA='25b412ff18c05e2607f4bc61b872f39826e8d53ffdc7647654c1f419e355d15f'
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()==SHA
model=YOLO(str(WEIGHTS));CLASSES=['D00','D10','D20','D40']
assert [model.names[i] for i in range(4)]==CLASSES
OUT=Path('/kaggle/working/country_error_run21');OUT.mkdir(exist_ok=True)
CONF=.46
protocol=dict(run=21,checkpoint='run17_best_finetuned.pt',sha256=SHA,confidence=CONF,imgsz=640,nms_iou=.7,max_det=300,matching_iou=.5,known_country_max_images=500,selection='ascending SHA256 of filename plus seed42; no outcome-based sampling',local_images_used=False,training_performed=False,threshold_retuned=False,US_role='country held out from our fine-tuning; external pretraining exposure unknown',China_role='development data reused; threshold optimized here',known_country_role='images outside team fine-tuning sample, countries represented in fine-tuning',uncertainty='single checkpoint; no verified route separation; near-duplicates unresolved')
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2))
(OUT/'environment.json').write_text(json.dumps(dict(ultralytics=ultralytics.__version__,torch=torch.__version__,device=str(DEVICE)),indent=2))
print('COUNTRY_MODEL_FROZEN',json.dumps(protocol),flush=True)
display(Markdown('**Setup interpretation.** The exact adopted fine-tuned model is verified. All countries use one frozen operating point. Country results cannot establish independent Saudi-road performance or exclude exposure during external pretraining.'))


## 2. Select and audit country cohorts
Keep all 1,977 China development images. Scan US test images and choose up to 500 remaining images for each fine-tuning country by a fixed filename hash. Exclude team training filenames and exact training hashes, retain country metadata, and validate every selected YOLO label. Save selected filenames, image and label hashes, dimensions, class support, exclusions and cohort roles before evaluation.

In [ ]:
train_manifest=pd.read_csv(BASE/'artifacts_run17/manifest.csv')
training=train_manifest.loc[train_manifest.split.eq('train')]
assert len(training)==8999
train_names=set(training.stem);train_hashes=set(training.sha256)
reference=train_manifest.loc[train_manifest.split.eq('val')]
assert len(reference)==1977
ref_hash=dict(zip(reference.stem,reference.sha256))
COUNTRIES=['China_MotorBike','United_States','Japan','India','Czech']
roles={'China_MotorBike':'development reused','United_States':'country held out from team fine-tuning','Japan':'image held out; country represented in fine-tuning','India':'image held out; country represented in fine-tuning','Czech':'image held out; country represented in fine-tuning'}
candidates={c:[] for c in COUNTRIES}
for p in sorted((BASE/'run17_team_v2_data/val/images').glob('*.jpg')):
    candidates['China_MotorBike'].append((p,BASE/'run17_team_v2_data/val/labels'/(p.stem+'.txt')))
for split in ['train','val','test_US']:
    for p in sorted((SOURCE/split/'images').glob('*.jpg')):
        if p.stem in train_names:continue
        country=next((c for c in COUNTRIES if p.stem.startswith(c+'_')),None)
        if split=='test_US':
            assert country in [None,'United_States'],p.name
            country='United_States'
        if country in ['Japan','India','Czech','United_States']:
            candidates[country].append((p,SOURCE/split/'labels'/(p.stem+'.txt')))
for country in candidates:
    unique={str(p):(p,l) for p,l in candidates[country]}
    candidates[country]=sorted(unique.values(),key=lambda item:hashlib.sha256(('42:'+item[0].stem).encode()).hexdigest())
assert len(candidates['China_MotorBike'])==1977
assert candidates['United_States'],'No US test images found; stop instead of inventing a held-out evaluation.'
seen_hash=set();selected=[];exclusions=[];cohorts=[];lookup={}
def audit_image(pair):
    p,l=pair;assert l.is_file(),f'Missing label {l}'
    digest=hashlib.sha256(p.read_bytes()).hexdigest()
    with Image.open(p) as im:im.load();w,h=im.size
    support=np.zeros(4,dtype=int)
    raw=l.read_text()
    for line in raw.splitlines():
        if not line.strip():continue
        v=list(map(float,line.split()));assert len(v)==5 and all(math.isfinite(x) for x in v),l
        c,x,y,bw,bh=v;assert c==int(c) and 0<=c<4 and 0<bw<=1 and 0<bh<=1,l
        assert x-bw/2>=-1e-4 and y-bh/2>=-1e-4 and x+bw/2<=1.0001 and y+bh/2<=1.0001,l
        support[int(c)]+=1
    return dict(image=str(p),label=str(l),stem=p.stem,sha256=digest,label_sha256=hashlib.sha256(raw.encode()).hexdigest(),width=w,height=h,**{c:int(support[i]) for i,c in enumerate(CLASSES)})
for country in COUNTRIES:
    limit=500 if country in ['Japan','India','Czech'] else len(candidates[country])
    accepted=[]
    # Audit in bounded chunks; selection does not depend on labels or predictions.
    for start in range(0,len(candidates[country]),128):
        if len(accepted)>=limit:break
        with ThreadPoolExecutor(max_workers=8) as pool:audited=list(pool.map(audit_image,candidates[country][start:start+128]))
        for row in audited:
            if len(accepted)>=limit:break
            if row['sha256'] in train_hashes:
                exclusions.append(dict(country=country,stem=row['stem'],reason='exact duplicate of fine-tuning image'));continue
            if row['sha256'] in seen_hash:
                exclusions.append(dict(country=country,stem=row['stem'],reason='exact duplicate within evaluation'));continue
            if country=='China_MotorBike':assert row['sha256']==ref_hash[row['stem']]
            seen_hash.add(row['sha256']);row.update(country=country,cohort_role=roles[country]);accepted.append(row)
        if start%1024==0:print('COUNTRY_AUDIT',country,len(accepted),flush=True)
    if country=='China_MotorBike':assert len(accepted)==1977,'China cohort changed; examine duplicate exclusion before proceeding.'
    if country=='United_States':assert accepted,'US cohort emptied by leakage checks.'
    selected.extend(accepted);cohorts.append(dict(country=country,role=roles[country],candidate_images=len(candidates[country]),selected_images=len(accepted),cap=limit))
    root=OUT/'cohorts'/country
    for folder in ['images','labels']:(root/folder).mkdir(parents=True,exist_ok=True)
    for row in accepted:
        for field,folder in [('image','images'),('label','labels')]:
            target=root/folder/Path(row[field]).name
            target.symlink_to(row[field])
        lookup[row['stem']]=row
    (root/'data.yaml').write_text(yaml.safe_dump(dict(path=str(root),train='images',val='images',names=dict(enumerate(CLASSES))),sort_keys=False))
manifest=pd.DataFrame(selected)
assert len(lookup)==len(manifest),'Repeated filename across countries; do not silently merge.'
manifest.to_csv(OUT/'evaluation_manifest.csv',index=False)
pd.DataFrame(exclusions,columns=['country','stem','reason']).to_csv(OUT/'exclusions.csv',index=False)
pd.DataFrame(cohorts).to_csv(OUT/'cohorts.csv',index=False)
(OUT/'manifest_sha256.txt').write_text(hashlib.sha256((OUT/'evaluation_manifest.csv').read_bytes()).hexdigest())
display(pd.DataFrame(cohorts));display(manifest.groupby('country')[CLASSES].sum())
display(Markdown(f'**Data interpretation.** Selected {len(manifest):,} images with validated labels and no exact overlap with team fine-tuning images. Saved exclusions: {len(exclusions)}. China remains reused development data; US is a held-out country from team fine-tuning. Other country cohorts, if nonempty, are held-out images from represented countries. Hash checks do not remove all near-duplicates.'))
print('COUNTRY_DATA_READY',json.dumps(cohorts),flush=True)


## 3. Evaluate country metrics and identify error candidates
For each nonempty cohort, run the official Ultralytics mAP evaluator with a low confidence floor, then collect deployment predictions at the frozen confidence 0.46. The separate operating-point evaluator matches predictions by descending confidence to the best unused same-class label at IoU >=0.5. Duplicates and wrong classes are false positives; unmatched labels are false negatives.

Error categories are geometric diagnostic candidates: duplicate, wrong-class overlap, localization mismatch, and background/annotation-review candidate. They do not prove a visual cause such as shadows or poor lighting. Save per-image counts and per-box error records for manual review. mAP uses its official evaluator and is distinct from these fixed-threshold counts.

In [ ]:
def ious(box,targets):
    if len(targets)==0:return np.empty(0)
    wh=np.maximum(0,np.minimum(box[2:],targets[:,2:])-np.maximum(box[:2],targets[:,:2]));inter=wh[:,0]*wh[:,1]
    a=max(0,box[2]-box[0])*max(0,box[3]-box[1]);b=(targets[:,2]-targets[:,0])*(targets[:,3]-targets[:,1])
    return inter/np.maximum(a+b-inter,1e-12)
def match(pred,gt):
    used=set();tp=np.zeros(4,int);fp=np.zeros(4,int);fn=np.zeros(4,int);errors=[]
    for p in pred[np.argsort(-pred[:,4],kind='stable')]:
        c=int(p[5]);ids=[i for i,g in enumerate(gt) if int(g[4])==c and i not in used];hit=None
        if ids:
            ov=ious(p[:4],gt[ids,:4]);j=int(np.argmax(ov))
            if ov[j]>=.5:hit=ids[j]
        if hit is not None:used.add(hit);tp[c]+=1;continue
        fp[c]+=1;overlap=ious(p[:4],gt[:,:4]);same=np.array([int(g[4])==c for g in gt],dtype=bool)
        if len(gt) and np.any((overlap>=.5)&same):kind='duplicate'
        elif len(gt) and np.any((overlap>=.5)&~same):kind='wrong_class_overlap'
        elif len(gt) and np.any((overlap>=.1)&same):kind='localization_candidate'
        else:kind='background_or_annotation_review'
        errors.append(dict(error='FP',damage_code=CLASSES[c],category=kind,confidence=float(p[4]),bbox_xyxy=p[:4].tolist()))
    for j,g in enumerate(gt):
        if j in used:continue
        c=int(g[4]);fn[c]+=1;ov=ious(g[:4],pred[:,:4]);same=np.array([int(p[5])==c for p in pred],dtype=bool)
        if len(pred) and np.any((ov>=.5)&~same):kind='wrong_class_overlap'
        elif len(pred) and np.any((ov>=.1)&same):kind='localization_candidate'
        else:kind='no_matching_detection_above_threshold'
        area=(g[2]-g[0])*(g[3]-g[1]);errors.append(dict(error='FN',damage_code=CLASSES[c],category=kind,confidence=None,bbox_xyxy=g[:4].tolist(),area_pixels=float(area)))
    return tp,fp,fn,errors
# Verify matching semantics before evaluating real images.
g=np.array([[0,0,10,10,3]],float)
a=match(np.array([[0,0,10,10,.9,3],[0,0,10,10,.8,3]],float),g);assert (a[0].sum(),a[1].sum(),a[2].sum())==(1,1,0)
a=match(np.array([[0,0,10,10,.9,0]],float),g);assert (a[0].sum(),a[1].sum(),a[2].sum())==(0,1,1)
a=match(np.empty((0,6)),g);assert a[2].sum()==1
assert np.allclose(ious(g[0,:4],g[:,:4]),[1])
country_rows=[];class_rows=[];image_rows=[];error_rows=[];examples={}
for cohort in cohorts:
    country=cohort['country'];n=cohort['selected_images']
    if n==0:continue
    print('COUNTRY_EVALUATION_START',country,n,flush=True)
    root=OUT/'cohorts'/country
    # Use a fresh model so validation cannot change deployment inference settings.
    evaluator=YOLO(str(WEIGHTS))
    val=evaluator.val(data=str(root/'data.yaml'),split='val',imgsz=640,batch=16,device=DEVICE,workers=2,conf=.001,iou=.7,max_det=300,plots=True,project=str(OUT/'official_evaluation'),name=country,exist_ok=False)
    official=dict(mAP50=float(val.box.map50),mAP50_95=float(val.box.map))
    ap={int(c):(float(val.box.ap50[i]),float(val.box.ap[i])) for i,c in enumerate(val.box.ap_class_index)}
    del evaluator,val;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    deployed=YOLO(str(WEIGHTS));tot_tp=np.zeros(4,int);tot_fp=np.zeros(4,int);tot_fn=np.zeros(4,int);processed=0;review=[]
    for r in deployed.predict(source=str(root/'images'),imgsz=640,conf=CONF,iou=.7,max_det=300,device=DEVICE,batch=1,stream=True,verbose=False):
        row=lookup[Path(r.path).stem];h,w=r.orig_shape;gt=[]
        for line in Path(row['label']).read_text().splitlines():
            if not line.strip():continue
            c,x,y,bw,bh=map(float,line.split());gt.append([(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h,c])
        gt=np.array(gt,float).reshape(-1,5)
        pred=np.column_stack([r.boxes.xyxy.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.boxes.cls.cpu().numpy()]).astype(float).reshape(-1,6)
        tp,fp,fn,errors=match(pred,gt);tot_tp+=tp;tot_fp+=fp;tot_fn+=fn;processed+=1
        key=row['stem'];image_rows.append(dict(country=country,stem=key,TP=int(tp.sum()),FP=int(fp.sum()),FN=int(fn.sum()),ground_truth=len(gt),predictions=len(pred)))
        for e in errors:
            if e['error']=='FN':e['area_fraction']=e['area_pixels']/(w*h)
            error_rows.append(dict(country=country,stem=key,**e))
        if errors:
            review.append(dict(stem=key,image=row['image'],gt=gt.tolist(),pred=pred.tolist(),errors=errors,FP=int(fp.sum()),FN=int(fn.sum())))
            # Keep the strongest error-count candidates, tie-broken by filename, for bounded memory.
            review=sorted(review,key=lambda x:(-(x['FP']+x['FN']),x['stem']))[:8]
        if processed%500==0:print('COUNTRY_PREDICTIONS',country,processed,flush=True)
    assert processed==n
    p=np.divide(tot_tp,tot_tp+tot_fp,out=np.zeros(4,float),where=(tot_tp+tot_fp)>0)
    rec=np.divide(tot_tp,tot_tp+tot_fn,out=np.zeros(4,float),where=(tot_tp+tot_fn)>0)
    f1=np.divide(2*tot_tp,2*tot_tp+tot_fp+tot_fn,out=np.zeros(4,float),where=(2*tot_tp+tot_fp+tot_fn)>0)
    supported=(tot_tp+tot_fn)>0
    for c,code in enumerate(CLASSES):class_rows.append(dict(country=country,damage_code=code,support=int(tot_tp[c]+tot_fn[c]),TP=int(tot_tp[c]),FP=int(tot_fp[c]),FN=int(tot_fn[c]),precision=float(p[c]),recall=float(rec[c]) if supported[c] else None,f1=float(f1[c]) if supported[c] else None,AP50=ap.get(c,(None,None))[0],AP50_95=ap.get(c,(None,None))[1]))
    TP=int(tot_tp.sum());FP=int(tot_fp.sum());FN=int(tot_fn.sum())
    result=dict(country=country,cohort_role=roles[country],images=n,**official,box_macro_f1_supported_classes=float(f1[supported].mean()) if supported.any() else None,supported_classes=int(supported.sum()),micro_precision=TP/(TP+FP) if TP+FP else 0.0,micro_recall=TP/(TP+FN) if TP+FN else None,micro_f1=2*TP/(2*TP+FP+FN) if 2*TP+FP+FN else None,TP=TP,FP=FP,FN=FN,false_positives_per_100_images=100*FP/n,pothole_recall=float(rec[3]) if supported[3] else None)
    country_rows.append(result);examples[country]=review
    pd.DataFrame(country_rows).to_csv(OUT/'country_comparison.csv',index=False);pd.DataFrame(class_rows).to_csv(OUT/'country_per_class.csv',index=False);pd.DataFrame(image_rows).to_csv(OUT/'per_image_errors.csv',index=False)
    (OUT/'error_records.json').write_text(json.dumps(error_rows));(OUT/'review_examples.json').write_text(json.dumps(examples))
    display(pd.DataFrame([result]));display(Markdown(f"**{country} interpretation.** {roles[country]}. Detection mAP50-95: {official['mAP50_95']:.2%}. At frozen confidence 0.46, there were {FP} false-positive boxes and {FN} missed labeled boxes across {n} images. Inspect per-class support before comparing aggregate scores. These figures do not establish Saudi-road generalization."))
    print('COUNTRY_EVALUATION_COMPLETE',json.dumps(result),flush=True)
    del deployed;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()


## 4. Summarize errors and country differences
Plot country metrics, export representative error galleries and report the largest failure categories and most missed classes. Galleries deliberately show high-error images, so their frequency is not representative. Green boxes are labels; red boxes are predictions with D codes and confidence. Read images together with their label-review caveat; geometric categories are not verified visual causes.

In [ ]:
import matplotlib.pyplot as plt
comparison=pd.DataFrame(country_rows);per_class=pd.DataFrame(class_rows);errs=pd.DataFrame(error_rows)
assert {'China_MotorBike','United_States'}<=set(comparison.country)
counts=errs.groupby(['country','error','category']).size().reset_index(name='boxes') if len(errs) else pd.DataFrame(columns=['country','error','category','boxes'])
counts.to_csv(OUT/'error_categories.csv',index=False)
missed=per_class.sort_values(['country','FN'],ascending=[True,False]).groupby('country',sort=False).head(1)
display(comparison);display(counts);display(missed[['country','damage_code','support','FN','recall']])
fig,ax=plt.subplots(figsize=(10,4));x=np.arange(len(comparison));ax.bar(x-.2,comparison.mAP50_95*100,.4,label='mAP50-95');ax.bar(x+.2,comparison.micro_f1*100,.4,label='Box micro-F1 at 0.46');ax.set_xticks(x,comparison.country,rotation=15);ax.set_ylabel('Score (%)');ax.set_title('Country diagnostics: different cohort roles and class mixtures');ax.legend();fig.tight_layout();fig.savefig(OUT/'country_comparison.png',dpi=150);plt.show();plt.close(fig)
for country,items in examples.items():
    if not items:continue
    items=items[:4];fig,axes=plt.subplots(2,2,figsize=(12,10));axes=np.array(axes).ravel()
    for ax in axes:ax.axis('off')
    for ax,item in zip(axes,items):
        im=Image.open(item['image']).convert('RGB');draw=ImageDraw.Draw(im);width=max(2,int(im.width/350))
        for g in item['gt']:draw.rectangle(g[:4],outline='lime',width=width);draw.text((g[0],max(0,g[1]-14)),'GT '+CLASSES[int(g[4])],fill='lime',stroke_width=1,stroke_fill='black')
        for p in item['pred']:draw.rectangle(p[:4],outline='red',width=width);draw.text((p[0],p[1]),CLASSES[int(p[5])]+f' {p[4]:.2f}',fill='red',stroke_width=1,stroke_fill='white')
        ax.imshow(im);ax.set_title(f"{item['stem']} | FP {item['FP']} FN {item['FN']}",fontsize=9)
    fig.suptitle(country+': high-error review candidates (green labels / red predictions)');fig.tight_layout();fig.savefig(OUT/(country+'_error_gallery.png'),dpi=130);display(fig);plt.close(fig)
china=comparison.loc[comparison.country.eq('China_MotorBike')].iloc[0];us=comparison.loc[comparison.country.eq('United_States')].iloc[0]
delta=(float(us.mAP50_95)-float(china.mAP50_95))*100
interpretations=[f"US mAP50-95 differs from China by {delta:+.2f} percentage points. China was used for development and threshold selection; US was not part of our fine-tuning. This descriptive gap can reflect domain, class mix and label differences; it is not a pure causal country effect."]
for _,r in missed.iterrows():interpretations.append(f"{r.country}: {r.damage_code} has the largest missed-box count ({int(r.FN)} of {int(r.support)} labels); recall {r.recall:.2%}. Counts depend on class support, so also inspect per-class recall.")
for country in comparison.country:
    sub=counts.loc[counts.country.eq(country)].sort_values('boxes',ascending=False)
    if len(sub):
        top=sub.iloc[0];interpretations.append(f"{country}: the largest geometric error category is {top['error']} / {top['category']} ({int(top['boxes'])} boxes). Review the gallery and original labels before assigning a visual cause.")
small_fn=sum(e['error']=='FN' and e.get('area_fraction',1)<.01 for e in error_rows);all_fn=sum(e['error']=='FN' for e in error_rows)
interpretations.append(f"{small_fn}/{all_fn} missed boxes cover less than 1% of their image. This is an error-size description, not proof that small size caused each miss.")
report=chr(10).join(['# Run 21: frozen-model country comparison and error analysis','',comparison.to_markdown(index=False),'',*interpretations,'','## Protocol and limits','','Exact adopted run17 checkpoint; confidence 0.46, resolution 640, NMS IoU 0.7, max_det 300. Official mAP uses confidence floor 0.001. Fixed-threshold counts use same-class descending-confidence greedy one-to-one matching at IoU >=0.5. Classes with no labels have undefined recall/F1 and are excluded from supported-class Macro-F1; inspect per-class support before comparing countries.','','No training, threshold tuning or local-image evaluation occurred. China is reused development data. US was excluded from team fine-tuning; external pretraining overlap remains unknown. Other country cohorts are image-held-out diagnostics from represented countries. Exact duplicate hashes were excluded; near-duplicates and route overlap remain unresolved. Galleries deliberately select high-error cases and are not representative. Error categories are geometric diagnostics, not verified visual causes. Adopted model and frozen settings remain unchanged. Local evaluation is pending.','','## Attribution','','RDD2022, Arya et al.; team YOLO conversion Sara. Published initialization SreekarAditya/yolo-rdd2022-benchmark, CC BY 4.0 according to publisher; retain applicable Ultralytics terms.'])
(OUT/'COUNTRY_ERROR_ANALYSIS.md').write_text(report)
summary=dict(status='complete',run=21,protocol=protocol,cohorts=cohorts,results=country_rows,per_class=class_rows,error_categories=counts.to_dict('records'),interpretation=interpretations,small_missed_boxes=small_fn,total_missed_boxes=all_fn,exclusions=len(exclusions),local_images_used=False,training_performed=False,threshold_retuned=False,US_evaluated=True,adopted_model_changed=False)
(OUT/'country_error_results.json').write_text(json.dumps(summary,indent=2))
display(Markdown('**Final interpretation.** '+' '.join(interpretations)+' Local independent testing remains pending. Keep the adopted model and confidence fixed.'))
print('COUNTRY_ERROR_ANALYSIS_COMPLETE',json.dumps(summary),flush=True)
